## Perform single cell quality control on the profiles

We utilize the conditions used in the ALSF project repository.
We detect poor quality nuclei segmentations.

NOTE: We run this notebook via `papermill` through this [bash script](./run_single_cell_qc.sh). 
We found it was easier to run through all 136 plates in this format over a for loop.

In [ ]:
import pathlib
import time

# Ignore FutureWarnings from cytodataframe due to skimage deprecation
# (does not affect functionality)
import warnings

import pandas as pd

from cosmicqc import find_outliers

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
plate_id = "SQ00014812"

In [3]:
# Parameters
plate_id = "SQ00015157"

In [4]:
# Directory containing the converted profiles
data_dir = pathlib.Path("/media/NVME_4TB/LINCS_cytotable_output/data/")

# Directory to save labeled data
labeled_dir = pathlib.Path(
    "/home/jenna/mnt/bandicoot/LINCS_data/processed_profiles/qc_profiles"
)
labeled_dir.mkdir(exist_ok=True)

# Create an empty dictionary to store data frames for each plate
all_qc_data_frames = {}

# Set the compartment of choice to perform QC at the start (will change later)
compartment = "Nuclei"

In [5]:
# Construct the file path for the given plate_id
file_path = data_dir / f"{plate_id}/{plate_id}.parquet"

if file_path.exists():
    start_time = time.time()  # Start timer for loading

    # Load the DataFrame with pandas
    plate_df = pd.read_parquet(file_path, engine="pyarrow")

    end_time = time.time()  # End timer for loading
    print(
        f"Loaded plate: {plate_id}, "
        f"Shape: {plate_df.shape}, "
        f"Time taken: {end_time - start_time:.2f} seconds"
    )
else:
    print(f"Parquet file for plate {plate_id} not found.")

Loaded plate: SQ00015157, Shape: (860747, 1821), Time taken: 16.36 seconds


In [6]:
# metadata columns to include in output data frame
metadata_columns = [
    "Image_Metadata_Plate",
    "Image_Metadata_Well",
    "Image_Metadata_Site",
    f"{compartment}_Location_Center_X",
    f"{compartment}_Location_Center_Y",
]

# Define the QC features
qc_features = [
    "Nuclei_Intensity_IntegratedIntensity_DNA",
    "Nuclei_AreaShape_Solidity",
    "Nuclei_Intensity_MassDisplacement_DNA",
]

# Filter plate_df to only include metadata columns and QC features
filtered_plate_df = plate_df[metadata_columns + qc_features]

# Drop any rows with NaN values in the QC features
filtered_plate_df = filtered_plate_df.dropna(subset=qc_features)

# Print the first few rows of the filtered DataFrame
print("Filtered plate DataFrame shape:", filtered_plate_df.shape)
filtered_plate_df.head()

Filtered plate DataFrame shape: (860622, 8)


,Image_Metadata_Plate,Image_Metadata_Well,Image_Metadata_Site,Nuclei_Location_Center_X,Nuclei_Location_Center_Y,Nuclei_Intensity_IntegratedIntensity_DNA,Nuclei_AreaShape_Solidity,Nuclei_Intensity_MassDisplacement_DNA
0,SQ00015157,A01,1,452.918455,48.909871,163.508697,0.958090,0.117379
1,SQ00015157,A01,1,348.545916,80.453161,149.192237,0.945669,1.438239
2,SQ00015157,A01,2,1773.529111,30.997957,67.376456,0.941799,0.937829
3,SQ00015157,A01,2,1242.572337,34.078696,101.006110,0.914909,1.045610
4,SQ00015157,A01,2,1213.263610,41.775072,48.470068,0.951602,1.340142


In [ ]:
# Find large nuclei outliers for the current plate
nuclei_clustered_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        # Set very low as to detect all instances of clustering nuclei
        "Nuclei_Intensity_MassDisplacement_DNA": 0.05,
        # Set higher than displacement to avoid false positives
        "Nuclei_Intensity_IntegratedIntensity_DNA": 1.5,
    },
)

# Convert to regular pandas DataFrame
nuclei_clustered_outliers = pd.DataFrame(nuclei_clustered_outliers)

print(nuclei_clustered_outliers.shape)
nuclei_clustered_outliers.sort_values(
    by="Nuclei_Intensity_MassDisplacement_DNA", ascending=True
).head(2)

Number of outliers: 25293 (2.94%)
Outliers Range:
Nuclei_Intensity_MassDisplacement_DNA Min: 0.797089075875017
Nuclei_Intensity_MassDisplacement_DNA Max: 17.791748426720016
Nuclei_Intensity_IntegratedIntensity_DNA Min: 295.3951463121921
Nuclei_Intensity_IntegratedIntensity_DNA Max: 3222.2013433622196
(25293, 7)


,Nuclei_Intensity_MassDisplacement_DNA,Nuclei_Intensity_IntegratedIntensity_DNA,Image_Metadata_Plate,Image_Metadata_Well,Image_Metadata_Site,Nuclei_Location_Center_X,Nuclei_Location_Center_Y
224925,0.797089,302.233732,SQ00015157,I15,9,397.193891,364.370784
372878,0.797172,297.427476,SQ00015157,L11,8,1365.378888,1209.534943


In [ ]:
# Find low nuclei solidity outliers for the current plate
solidity_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        # Set at this point where it looks like it starts to detect good quality nuclei
        "Nuclei_AreaShape_Solidity": -1.6,
    },
)

# Convert to regular pandas DataFrame
solidity_nuclei_outliers = pd.DataFrame(solidity_nuclei_outliers)

print(solidity_nuclei_outliers.shape)
solidity_nuclei_outliers.sort_values(
    by="Nuclei_AreaShape_Solidity", ascending=False
).head(2)

Number of outliers: 64700 (7.52%)
Outliers Range:
Nuclei_AreaShape_Solidity Min: 0.4979757085020242
Nuclei_AreaShape_Solidity Max: 0.9236227824463119
(64700, 6)


,Nuclei_AreaShape_Solidity,Image_Metadata_Plate,Image_Metadata_Well,Image_Metadata_Site,Nuclei_Location_Center_X,Nuclei_Location_Center_Y
263001,0.923623,SQ00015157,N23,6,154.312576,536.883542
762618,0.923623,SQ00015157,I15,5,241.862263,1661.844342


In [9]:
# Set compartment as cells
compartment = "Cells"

# metadata columns to include in output data frame
metadata_columns = [
    "Image_Metadata_Plate",
    "Image_Metadata_Well",
    "Image_Metadata_Site",
    f"{compartment}_Location_Center_X",
    f"{compartment}_Location_Center_Y",
]

# Define the QC features
qc_features = ["Cells_Intensity_IntegratedIntensity_DNA"]

# Filter plate_df to only include metadata columns and QC features
filtered_plate_df = plate_df[metadata_columns + qc_features]

# Drop any rows with NaN values in the QC features
filtered_plate_df = filtered_plate_df.dropna(subset=qc_features)

In [ ]:
# Find cell outliers for the current plate
cell_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        # Set low to attempt to detect all instances of abnormally high int in nuclei
        # for whole cells
        "Cells_Intensity_IntegratedIntensity_DNA": 0.5,
    },
)

# Convert to regular pandas DataFrame
cell_outliers = pd.DataFrame(cell_outliers)

print(cell_outliers.shape)
cell_outliers.sort_values(
    by="Cells_Intensity_IntegratedIntensity_DNA", ascending=True
).head(2)

Number of outliers: 207868 (24.15%)
Outliers Range:
Cells_Intensity_IntegratedIntensity_DNA Min: 314.7497231019661
Cells_Intensity_IntegratedIntensity_DNA Max: 11241.166449594311


(207868, 6)


,Cells_Intensity_IntegratedIntensity_DNA,Image_Metadata_Plate,Image_Metadata_Well,Image_Metadata_Site,Cells_Location_Center_X,Cells_Location_Center_Y
156320,314.749723,SQ00015157,E16,9,884.329857,871.781280
387259,314.749753,SQ00015157,J06,2,634.114318,586.241531


In [11]:
# Add QC failure columns to plate_df based on outlier indices
plate_df["cqc.failed_clustered_nuclei"] = plate_df.index.isin(
    nuclei_clustered_outliers.index
)
plate_df["cqc.failed_low_solidity_nuclei"] = plate_df.index.isin(
    solidity_nuclei_outliers.index
)
plate_df["cqc.failed_cell_outlier"] = plate_df.index.isin(cell_outliers.index)

# Save the labeled dataframe to parquet
labeled_path = labeled_dir / f"{plate_id}_qc_labeled.parquet"
plate_df.to_parquet(labeled_path, index=False)
print(f"Labeled dataframe for plate {plate_id} saved to {labeled_path}")

# print the shape and head of the updated DataFrame
print(plate_df.shape)
plate_df.head()

Labeled dataframe for plate SQ00015157 saved to /home/jenna/mnt/bandicoot/LINCS_data/processed_profiles/qc_profiles/SQ00015157_qc_labeled.parquet
(860747, 1824)


,Metadata_ImageNumber,Image_Metadata_Col,Image_Metadata_Plate,Image_Metadata_Row,Image_Metadata_Site,Image_Metadata_Well,Metadata_ObjectNumber,Metadata_ObjectNumber_1,Metadata_ObjectNumber_2,Image_Count_Cells,...,Nuclei_Texture_Variance_ER_5_0,Nuclei_Texture_Variance_Mito_10_0,Nuclei_Texture_Variance_Mito_20_0,Nuclei_Texture_Variance_Mito_5_0,Nuclei_Texture_Variance_RNA_10_0,Nuclei_Texture_Variance_RNA_20_0,Nuclei_Texture_Variance_RNA_5_0,cqc.failed_clustered_nuclei,cqc.failed_low_solidity_nuclei,cqc.failed_cell_outlier
0,1,1,SQ00015157,1,1,A01,1,1.0,1.0,92.0,...,1.819299,2.077875,1.663997,2.372667,3.304246,3.834013,3.069262,False,False,False
1,1,1,SQ00015157,1,1,A01,2,2.0,2.0,92.0,...,2.204496,1.648711,1.813057,1.507026,2.271174,2.267891,2.393204,False,False,False
2,2,1,SQ00015157,1,2,A01,1,1.0,1.0,334.0,...,1.673851,0.494586,0.228945,1.001797,2.036476,2.373264,2.322208,False,False,False
3,2,1,SQ00015157,1,2,A01,2,2.0,2.0,334.0,...,2.138700,2.143279,2.582343,2.033866,3.294455,3.532377,3.197854,False,True,False
4,2,1,SQ00015157,1,2,A01,3,3.0,3.0,334.0,...,1.702788,2.678977,1.089011,3.796997,0.544414,0.057918,1.756489,False,False,False
